<a href="https://colab.research.google.com/github/amathe1/Neural_Networks/blob/main/churn_risk_of_customers_LightningModule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem statement :**

**A telecom company wants to predict the customer churn risk using customer usage, billing and contract data.**

The model must classify each customer into three risk categories :
- 0 Low churn risk
- 1 Medium churn risk
- 2 High churn risk

In [ ]:
#Step1 - Load and prepare data
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [ ]:
# Creating a dataframe to read input data from below cv files
df = pd.read_csv('/content/sample_data/customer_churn_synthetic_1000.csv')
X = df.drop("churn", axis=1).values
y = df["churn"].values


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
!pip install lightning

In [ ]:

import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# -------------------------
# Lightning Model
# -------------------------

class ChurnModel(L.LightningModule):
    def __init__(self, input_dim, hidden_dim=64, lr=0.001):
        super().__init__()
        self.save_hyperparameters()

        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 3)   # 3 output classes (0,1,2)
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


In [ ]:
# Convert to tensors (if not already)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)


In [ ]:
model = ChurnModel(input_dim=X_train.shape[1])

trainer = L.Trainer(
    max_epochs=50,
    accelerator="auto",     # GPU if available
    devices="auto",
    log_every_n_steps=10
)

trainer.fit(model, train_loader, val_loader)
